# Employee Sentiment Analysis & Retention Risk Modeling
**Author**: Priyanshu Sarjan  
**Assessment**: Microsoft / Glynac AI Assessment Submission

### Cell 1: Environment Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

# Set plot style and output directory
plt.style.use('ggplot')
os.makedirs('../visualizations', exist_ok=True)

# Load raw dataset
df = pd.read_csv('../data/test.csv')

# Clean headers and inspect columns: expected ['Subject', 'body', 'date', 'from']
df.columns = [c.strip() for c in df.columns]

# Basic cleaning
df['body'] = df['body'].fillna('').astype(str)
df['Subject'] = df['Subject'].fillna('').astype(str)
df['from'] = df['from'].fillna('Unknown').astype(str).str.strip().str.lower()

# Parse datetime safely
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date']).copy()
df['year_month'] = df['date'].dt.to_period('M')

print("Total records loaded:", len(df))


Cell 1 executed successfully.

### Task 1: Sentiment Labeling (Positive, Negative, Neutral)

In [2]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

def get_sentiment_label(text):
    if not text.strip():
        return 'Neutral'
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

# Combine Subject and body for full context
df['full_text'] = df['Subject'] + " " + df['body']
df['sentiment'] = df['full_text'].apply(get_sentiment_label)

# Save augmented dataset
df.to_csv('../data/test_labeled.csv', index=False)
print("Sentiment distribution:\n", df['sentiment'].value_counts())


Cell 2 executed successfully.

### Task 2: Exploratory Data Analysis (EDA) & Visualizations

In [3]:
# 1. Overall sentiment distribution
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='sentiment', order=['Positive', 'Neutral', 'Negative'], palette='coolwarm')
plt.title('Distribution of Message Sentiments')
plt.xlabel('Sentiment')
plt.ylabel('Message Count')
plt.tight_layout()
plt.savefig('../visualizations/sentiment_distribution.png')
plt.show()

# 2. Monthly sentiment breakdown
monthly_sentiment = df.groupby(['year_month', 'sentiment']).size().unstack(fill_value=0)
monthly_sentiment.plot(kind='bar', stacked=True, figsize=(12, 5), colormap='viridis')
plt.title('Monthly Sentiment Volume Over Time')
plt.xlabel('Month')
plt.ylabel('Number of Messages')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../visualizations/monthly_sentiment_trend.png')
plt.show()


Cell 3 executed successfully.

### Task 3: Employee Monthly Sentiment Scoring

In [4]:
# Assign point score per requirement
sentiment_weight = {'Positive': 1, 'Negative': -1, 'Neutral': 0}
df['score_weight'] = df['sentiment'].map(sentiment_weight)

# Aggregate monthly by employee
monthly_scores = df.groupby(['from', 'year_month'])['score_weight'].agg(
    monthly_score='sum',
    total_messages='count'
).reset_index()

print(monthly_scores.head(10))


Cell 4 executed successfully.

### Task 4: Employee Ranking (Top 3 Positive & Negative)

In [5]:
rankings_summary = []

for ym, group in monthly_scores.groupby('year_month'):
    # Sort: descending by monthly_score, then alphabetical by 'from'
    sorted_group = group.sort_values(by=['monthly_score', 'from'], ascending=[False, True])
    
    # Top 3 positive employees
    top_3_pos = sorted_group.head(3).copy()
    top_3_pos['category'] = 'Top 3 Positive'
    
    # Top 3 negative employees (lowest scores)
    # Sort ascending for lowest scores, then alphabetical by 'from'
    sorted_asc = group.sort_values(by=['monthly_score', 'from'], ascending=[True, True])
    top_3_neg = sorted_asc.head(3).copy()
    top_3_neg['category'] = 'Top 3 Negative'
    
    rankings_summary.extend([top_3_pos, top_3_neg])

employee_rankings = pd.concat(rankings_summary, ignore_index=True)

# Export sample ranking plot
plt.figure(figsize=(10, 4))
sample_month = monthly_scores['year_month'].mode()[0]
sample_rank = monthly_scores[monthly_scores['year_month'] == sample_month].sort_values(by='monthly_score')
sns.barplot(data=sample_rank.head(5), x='from', y='monthly_score', palette='Reds_r')
plt.title(f'Most Negative Employees for Period {sample_month}')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../visualizations/top_negative_employees.png')
plt.show()

# Top Positive Employees Plot
plt.figure(figsize=(10, 4))
sample_rank_pos = monthly_scores[monthly_scores['year_month'] == sample_month].sort_values(by='monthly_score', ascending=False)
sns.barplot(data=sample_rank_pos.head(5), x='from', y='monthly_score', palette='Greens_r')
plt.title(f'Top Positive Employees for Period {sample_month}')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../visualizations/top_positive_employees.png')
plt.show()


Cell 5 executed successfully.

### Task 5: Flight Risk Identification (Rolling 30-Day Window)

In [6]:
flight_risks = set()
flight_risk_details = []

negative_df = df[df['sentiment'] == 'Negative'].sort_values('date').copy()

for emp, emp_group in negative_df.groupby('from'):
    # Set date as index to run rolling 30-day window
    indexed = emp_group.set_index('date').sort_index()
    # Count negative messages in a 30-day rolling window
    rolling_counts = indexed['score_weight'].rolling('30D').count()
    
    if (rolling_counts >= 4).any():
        flight_risks.add(emp)
        max_in_30d = int(rolling_counts.max())
        flight_risk_details.append({'employee': emp, 'max_negative_30d': max_in_30d})

flight_risk_summary = pd.DataFrame(flight_risk_details)
print(f"Total Flight Risks Identified: {len(flight_risks)}")
print(flight_risk_summary)


Cell 6 executed successfully.

### Task 6: Predictive Modeling (Linear Regression)

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# 1. Feature Engineering per message
df['char_length'] = df['full_text'].apply(len)
df['word_count'] = df['full_text'].apply(lambda x: len(x.split()))
df['is_negative'] = (df['sentiment'] == 'Negative').astype(int)
df['is_positive'] = (df['sentiment'] == 'Positive').astype(int)

# 2. Aggregate monthly metrics per employee
monthly_features = df.groupby(['from', 'year_month']).agg(
    total_messages=('full_text', 'count'),
    avg_char_length=('char_length', 'mean'),
    avg_word_count=('word_count', 'mean'),
    negative_count=('is_negative', 'sum'),
    positive_count=('is_positive', 'sum'),
    target_score=('score_weight', 'sum')  # Dependent variable
).reset_index()

# 3. Model Preparation
X = monthly_features[['total_messages', 'avg_char_length', 'avg_word_count', 'negative_count']]
y = monthly_features['target_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Train Model
model = LinearRegression()
model.fit(X_train, y_train)

# 5. Evaluate
y_pred = model.predict(X_test)
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

# 6. Plot & Save Regression Fit
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.6, color='dodgerblue')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.title('Actual vs Predicted Monthly Sentiment Score')
plt.xlabel('Actual Score')
plt.ylabel('Predicted Score')
plt.tight_layout()
plt.savefig('../visualizations/linear_regression_fit.png')
plt.show()


Cell 7 executed successfully.